# WOVO — Remote Ollama Server on Colab

This notebook turns Google Colab's free T4 GPU into a remote Ollama server for the WOVO pipeline.

**Before running:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Get a free ngrok auth token from https://dashboard.ngrok.com/signup
3. Paste it in Cell 3 below

**After running all cells:**
Copy the printed `OLLAMA_BASE_URL` into your `.env.local` and run the pipeline normally.

In [ ]:
#@title Cell 1 — Check GPU & Install Ollama
!nvidia-smi
print("\n" + "="*50)
print("Installing Ollama...")
print("="*50)
!curl -fsSL https://ollama.ai/install.sh | sh
!ollama --version

In [ ]:
#@title Cell 2 — Start Ollama & Pull Models
import subprocess
import time

# Start Ollama server in background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Pull the 3 models used by WOVO pipeline jobs
models = [
    ("qwen3:4b",  "Text generation — survey analysis, clustering labels, forecasts, voice analytics"),
    ("gemma3:1b", "Lightweight — payslip anomaly interpretation"),
    ("bge-m3",    "Embeddings — case clustering (768-dim vectors)"),
]

for model_name, purpose in models:
    print(f"\n{'='*50}")
    print(f"Pulling {model_name} — {purpose}")
    print(f"{'='*50}")
    !ollama pull {model_name}

print("\n✅ All models ready!")
!ollama list

In [ ]:
#@title Cell 3 — Expose via ngrok

# ============================================================
# PASTE YOUR NGROK AUTH TOKEN HERE
# Get one free at: https://dashboard.ngrok.com/signup
NGROK_AUTH_TOKEN = ""  # <-- paste your token
# ============================================================

if not NGROK_AUTH_TOKEN:
    raise ValueError(
        "Please set NGROK_AUTH_TOKEN above. "
        "Get a free token at https://dashboard.ngrok.com/signup"
    )

!pip install -q pyngrok
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(11434)
public_url = tunnel.public_url

print(f"\n{'='*60}")
print(f"  Ollama is live! Set this in your .env.local:")
print(f"")
print(f"  OLLAMA_BASE_URL={public_url}/api")
print(f"")
print(f"  Quick test from your terminal:")
print(f"  curl {public_url}/api/tags")
print(f"{'='*60}")

In [ ]:
#@title Cell 4 — Keep Alive (run this and leave it)
import time
import requests

print("Keeping Colab session alive. Leave this cell running.")
print("Press the stop button (or Runtime → Interrupt) to shut down.\n")

tick = 0
while True:
    tick += 1
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        model_count = len(r.json().get("models", []))
        status = f"OK ({model_count} models loaded)"
    except Exception as e:
        status = f"ERROR: {e}"

    mins = tick
    print(f"[{mins:>4}m] Ollama health: {status}", end="\r")
    time.sleep(60)